# House Price Prediction Model Data Preprocessing Notebook

This notebook prepares the raw housing data for modeling.

**Goals**

- Load the raw CSV file and inspect basic properties.
- Clean and standardize column types.
- Handle missing values and simple outliers.
- Build a reusable preprocessing pipeline (using `scikit-learn`) that:
  - Scales numeric features.
  - Imputes missing values.
  - One-hot encodes categorical features.
- Split the data into train / validation / test sets and save:
  - The splits as CSV files.
  - The fitted preprocessing pipeline for later use in the modeling notebook.

In [2]:
# import libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import joblib
import os
import random

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)


## 1. Load Raw Data


In [5]:
DATA_DIR = '../'
os.makedirs('data', exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, 'california_housing.csv')

df = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df)} rows of data from {CSV_PATH}")

print(df.shape)
df.head()


Loaded 227215 rows of data from ../california_housing.csv
(227215, 12)


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,102093.0,for_sale,3.000000e+06,7.0,6.0,NaN,1760549.0,Balzola,California,0.0,6889.0,NaN
1,23826.0,for_sale,2.147484e+09,2.0,2.0,0.12,11355.0,International,California,NaN,885.0,NaN
2,98034.0,for_sale,1.000000e+07,NaN,NaN,123.97,NaN,Playa de Novillero,California,NaN,NaN,NaN
3,16829.0,for_sale,3.280000e+05,3.0,3.0,NaN,915973.0,Quintana Roo,California,NaN,NaN,NaN
4,17458.0,for_sale,1.990000e+05,NaN,NaN,5.10,2937.0,Milford,California,96121.0,NaN,NaN


## 2. Initial Inspection

Check column types, missing values, and basic statistics to understand the data.


In [6]:
df.info()
df.describe(include='all').transpose()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227215 entries, 0 to 227214
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   brokered_by     226564 non-null  float64
 1   status          227215 non-null  object 
 2   price           227182 non-null  float64
 3   bed             196146 non-null  float64
 4   bath            194142 non-null  float64
 5   acre_lot        204237 non-null  float64
 6   street          226421 non-null  float64
 7   city            227097 non-null  object 
 8   state           227215 non-null  object 
 9   zip_code        227194 non-null  float64
 10  house_size      197617 non-null  float64
 11  prev_sold_date  193018 non-null  object 
dtypes: float64(8), object(4)
memory usage: 20.8+ MB


brokered_by         651
status                0
price                33
bed               31069
bath              33073
acre_lot          22978
street              794
city                118
state                 0
zip_code             21
house_size        29598
prev_sold_date    34197
dtype: int64

## 3. Drop Invalid or Incomplete Rows

We remove any rows that contain missing values in key modeling columns:
`price, bed, bath, acre_lot, street, city, state, zip_code, house_size`.

In [8]:
required_cols = ["price", "bed", "bath", "acre_lot", "street", "city", "state", "zip_code", "house_size"]

# Keep only columns that exist in the CSV
required_cols = [c for c in required_cols if c in df.columns]

# Drop rows missing any required field
df = df.dropna(subset=required_cols).reset_index(drop=True)

print("Shape after dropping invalid rows:", df.shape)
df.head()


Shape after dropping invalid rows: (170573, 12)


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,4311,for_sale,199900.0,3.0,1.0,0.18,1466188.0,Blythe,California,92225,1014.0,NaT
1,4311,for_sale,172999.0,3.0,2.0,0.16,987585.0,Blythe,California,92225,1132.0,1984-06-29
2,64877,for_sale,79900.0,4.0,2.0,0.16,1533451.0,Blythe,California,92225,1272.0,NaT
3,54422,for_sale,69000.0,3.0,1.0,0.91,1626662.0,Blythe,California,92225,1134.0,NaT
4,109780,for_sale,75000.0,3.0,2.0,0.33,540514.0,Blythe,California,92225,1248.0,NaT


## 4. Filter Out Extreme Price Outliers

Even if rows are dropped, we still cap extreme price values at the 99th percentile so the model does not learn from impossible values.


In [9]:
price_cap = df["price"].quantile(0.99)
df["price"] = df["price"].clip(upper=price_cap)

print("99th percentile price cap:", price_cap)
print("Final cleaned data shape:", df.shape)


99th percentile price cap: 6713999.999999942
Final cleaned data shape: (170573, 12)


## 5. Save Final Cleaned Output

In [10]:
df.to_csv("final_housing_clean.csv", index=False)
print("Final cleaned dataset saved.")

Final cleaned dataset saved.


## 6. Train / Validation / Test Split

We split the data into:

- 70% training
- 15% validation
- 15% test

The model notebook will use the training and validation sets for fitting and tuning,
and the test set will be reserved for final evaluation.

In [11]:
# Features and target
feature_cols = [col for col in df.columns if col not in [TARGET_COL]]

X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

# First split: train vs (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED
)

# Second split: validation vs test (equal split of the temp set)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)


Train shape: (119401, 11)
Val shape: (25586, 11)
Test shape: (25586, 11)


## 7. Save Processed Data



In [19]:
SAVE_DIR = "processed"
os.makedirs(SAVE_DIR, exist_ok=True)

train_df = X_train.copy()
train_df["price"] = y_train.values

val_df = X_val.copy()
val_df["price"] = y_val.values

test_df = X_test.copy()
test_df["price"] = y_test.values

train_df.to_csv(os.path.join(SAVE_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(SAVE_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(SAVE_DIR, "test.csv"), index=False)

print("Saved splits with target column included.")


Saved splits with target column included.


## 8. Summary

- The dataset was successfully loaded from a local CSV file using pandas.
- Key numeric columns such as `price, bed, bath, acre_lot, street, zip_code, house_size` were validated for type consistency.
- Invalid or incomplete rows containing missing values in modeling-critical fields were removed, resulting in a cleaner dataset for training.
- Train, validation, and test splits were created using a reproducible 70/15/15 partition.
- The raw splits were saved into separate CSV files for portability and later reuse.
- The preprocessing phase produced reusable artifacts that the model training notebook can reload without re-processing the data.